# D1.1 · The sensor estate — EDR, DLP, CSPM and CNAPP against an agent

**Function D — The Agentic SOC → Discover — the Sensors, and the Agent-Shaped Hole in Them**

Builds on **[D1.0 · Start here — the agentic SOC, and the stack that runs it](https://spbreed.github.io/cyber-commons/lessons/D1.0.html)**.

| | |
|---|---|
| Tools used | Wazuh, Prowler, Falco |

## What this lesson is

**What it covers.** Scoring the four sensor classes an estate already owns — EDR, DLP, CSPM, CNAPP — against nine things an agent does in an ordinary day, and reading the actions no class sees at all.

**Why a security engineer needs it.** "We already have visibility" is the most common answer to an agent detection roadmap and it is answerable with a matrix rather than an opinion. The distinction that makes it honest is visibility rather than alerting: a sensor that is not in the path cannot be tuned into one that is, so the uncovered rows are an architecture finding and not a backlog item.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Four security products already watch the estate and every one of them was bought for a person on a host. Score them against what an agent actually does and four of its nine ordinary actions are seen by nothing at all — not seen badly, not alerted on late, simply outside the field of view.

> **At CyberTravels.** CyberTravels bought all four before it shipped an agent, and all four still work. What none of them is in the path of is the Workflow Agent reading a booking through the internal API, putting it in a prompt, calling the vendor MCP server and issuing the refund — which is the entire incident, start to finish, invisible to the estate's whole security stack.

## 2 · The framework

```
   what already watches the estate      what an agent does

   EDR    Wazuh agent          -----> writes a file        ##
   CNAPP  Falco + Trivy        -----> spawns a child       ##
   EDR    (partial)            -----> opens TLS to a model ..
   CSPM   Prowler              -----> assumes an IAM role  ..
   DLP    regex + FIM          -----> exfil to allowed SaaS ..

                                      reads a customer record
                                      puts it in a prompt
                                      calls a vendor MCP tool     <- NOTHING
                                      issues a refund

   the four uncovered rows are not badly tuned. no sensor class
   is in the path. they all happen inside the reasoning loop, or
   behind an API the host never observes.
```

Nobody starts an agentic SOC from nothing. Four classes of sensor are already
deployed, already paid for, and already producing alerts:

| class | what it watches | open-source reference |
|---|---|---|
| **EDR** | processes, files and network on a host | Wazuh agent |
| **DLP** | sensitive content leaving a monitored channel | regex + file integrity monitoring |
| **CSPM** | cloud configuration, evaluated on a schedule | Prowler, ScoutSuite |
| **CNAPP** | container runtime and image contents | Falco, Trivy |

All four work. The question is not whether they are good — it is **which parts
of an agent's working day they are in the path of at all.**

That is a matrix, and it is worth building before a roadmap rather than after,
because the answer decides whether the next quarter is spent tuning or spent
building a source that does not exist yet.

The distinction that makes the matrix honest is **visibility, not alerting**.
"Would this fire" is a tuning question. "Is this sensor in the path" is a fact
about architecture, and a sensor that is not in the path cannot be tuned into
one that is.

Read the uncovered rows rather than the percentage. If they are arbitrary, tune.
If they share a property — every one inside the reasoning loop, or behind an API
the host never observes — then a fifth product of the same four kinds will not
move them.

> **Anchor → D1.0.** Discover has a floor, and this lesson measures it. Four of the nine actions have no sensor in their path at all, so for those the interval is not long — it is undefined, and no amount of tuning the four products you own changes that.

## 3 · The procedure, as a skill

The skill scores four sensor classes against nine things CyberTravels' agents do in an ordinary day, takes the union per row rather than summing the columns, and prints the actions no class sees at all.

### The skill — [`skills/detection/sensor-coverage-matrix/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/sensor-coverage-matrix/SKILL.md)

```yaml
name: sensor-coverage-matrix
description: >-
  Score the sensor classes an estate already owns — EDR, DLP, CSPM, CNAPP —
  against what an agent actually does, and name the actions no sensor sees. Use
  before buying detection tooling for an agent estate, or when someone claims
  the estate is already covered.
allowed-tools: Read, Grep, Glob
```

# Coverage is a column, not a percentage

Every estate that runs agents already owns four classes of sensor, bought for
people and hosts. The question is not whether they work — they do, at what they
were built for — but which parts of an agent's working day are inside their
field of view at all.

Answer it as a matrix and one number falls out that no product page will give
you: the actions seen by **nothing**.

## When to use this

Before a detection roadmap, before a tooling purchase, and any time "we already
have visibility" is offered as an answer about an agent estate.

## Procedure

**1 — List the sensor classes you own, with the product behind each.** Naming
the product keeps the conversation checkable. The open-source references are
Wazuh for EDR, regex plus file integrity monitoring for the DLP that most teams
actually have, Prowler or ScoutSuite for CSPM, and Falco plus Trivy for CNAPP.

**2 — List what the agents do, not what you fear.** An ordinary day: files
written, an outbound session to a model API, a customer record read through an
internal API, tokens placed in a prompt, a vendor tool called, money moved, a
role assumed, a child process spawned.

**3 — Score each cell as full, partial or none — on visibility, not alerting.**
"Would this sensor alert" is a tuning question and comes later. "Is this sensor
in the path" is a fact about architecture, and it is the one that decides
whether tuning is even possible.

**4 — Read the uncovered rows, and read what they have in common.** If the
uncovered set is arbitrary, tune. If it is coherent — every action inside the
reasoning loop, or behind an API the host never observes — then a fifth product
of the same four kinds will not move it, and the answer is a new source rather
than a better rule.

## Example

```
per-sensor coverage of the agent's day
  EDR     3.0 / 9   33%
  DLP     0.5 / 9   6%
  CSPM    0.5 / 9   6%
  CNAPP   3.0 / 9   33%

  best single sensor   EDR
  all four combined    3.5 / 9   39%
```

The run continues past this. The script is the example: `test_skills.py`
executes it on every build, so this block cannot drift from what the skill
actually prints.

## Output contract

```json
{
  "sensors": [{"name": "str", "product": "str", "sees": "str"}],
  "actions": [{"action": "str", "coverage": {"EDR": "full|partial|none"}}],
  "per_sensor": [{"name": "str", "score": 0.0, "of": 0}],
  "combined": {"score": 0.0, "of": 0},
  "uncovered": ["str"]
}
```

## Failure modes

- **Scoring alerting rather than visibility.** A sensor that is not in the path
  cannot be tuned into one that is.
- **Summing the columns.** Four sensors covering the same three actions is
  still three actions; take the union per row, which is what the script does.
- **Reporting the combined percentage alone.** 39% sounds like a tuning problem.
  The four named rows are the finding.
- **Treating a partial as a full.** Seeing a TLS session open is not seeing what
  crossed it.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/sensor-coverage-matrix/scripts/sensor_coverage_matrix.py
SCRIPT = "skills/detection/sensor-coverage-matrix/scripts/sensor_coverage_matrix.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

EDR and CNAPP each cover about a third of the agent's day, DLP and CSPM almost none of it, and all four combined still leave four of the nine actions seen by nothing: reading a customer record through an internal API, placing it in a prompt, calling a vendor MCP tool, and issuing a refund. Those four share a property, and it is the argument for D1.3 and D2.1 rather than for a fifth product.

## Your turn

Build the same matrix for your estate with your own actions in the rows. The number that matters is not the percentage — it is whether the uncovered rows have something in common.

---

**Next → [D1.2 · Drift monitoring — behaviour that changes without a code change](https://spbreed.github.io/cyber-commons/lessons/D1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*